# TxGNN — Per-Block Gumbel Disease-Similarity Gate

Evaluation notebook for the **learned, per-block similarity gate** that replaces the fixed
exponential gate `c_i = 0.7·exp(-0.7·deg_i) + 0.2`.

It trains and evaluates up to three architecture *arms* under an identical protocol, so the
numbers drop straight into the same tables as
[`TxGNN_PyG_Train_Eval.ipynb`](TxGNN_PyG_Train_Eval.ipynb):

| arm | `agg_measure` | blocks | gate | isolates |
|---|---|---|---|---|
| **A** baseline | `rarity` | 2 (disease-disease, gene/protein) | fixed `0.7·exp(-0.7·deg)+0.2` | — |
| **B** more signal | `rarity_4block` | 4 (+ phenotype, exposure) | *same* fixed gate, fixed mean over blocks | "more signal" |
| **C** learned gate | `gumbel_block` | 4 | learned per-block binary Gumbel-Softmax | "smarter gating" |

B exists so that any gain in C can be attributed to the *gate* rather than to the two extra
signature blocks, which the baseline never used at all.

---

## Reusing your existing weights

Two independent kinds of reuse, both on by default:

1. **Pretrained encoder** (§4). Self-supervised pretraining is the expensive stage (~19 min/seed)
   and it is *architecture-independent*: `pretrain_mode=True` short-circuits before the similarity
   block, so the gate receives no gradient and the encoder produced is identical for all three
   arms. All arms therefore start from one shared encoder — which both saves time and removes
   pretraining variance as a confound. The notebook reuses the checkpoint your baseline run
   already wrote, and only pretrains if none is found.
2. **Baseline results** (§7). Arm A's numbers are read from `results/TxGNN_<split>_seed<n>_full/`
   if present, so you do not retrain the baseline at all.

## What this notebook does *not* do

It does not touch the baseline notebook or its outputs. Results land in
`results/gumbel/<arm>_<split>_seed<n>/`, checkpoints in `saved_models/gumbel/`.

## 1 · Environment

In [ ]:
import os, sys, json, time, copy, pickle, random, shutil, platform, warnings
from pathlib import Path
from collections import OrderedDict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

def _find_repo_root(start=None):
    p = Path(start or os.getcwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / 'pyg_implementation' / 'txgnn').is_dir():
            return cand
    raise RuntimeError('Could not find pyg_implementation/txgnn above ' + str(p))

REPO_ROOT = _find_repo_root()
PYG_ROOT  = REPO_ROOT / 'pyg_implementation'
sys.path.insert(0, str(PYG_ROOT))

import torch
import torch_geometric
import txgnn
from txgnn import TxData, TxGNN, TxEval

assert 'pyg_implementation' in txgnn.__file__, (
    'Wrong txgnn package on sys.path (%s). Restart the kernel.' % txgnn.__file__)

# The per-block machinery. Import fails loudly if the model.py changes are not in place.
from txgnn.model import (DISEASE_SIM_BLOCKS, BLOCK_ETYPES, BLOCK_NODES,
                         N_SIM_BLOCKS, BLOCK_AGG_MEASURES, VALID_AGG_MEASURES)

print('repo root     :', REPO_ROOT)
print('txgnn package :', txgnn.__file__)
print('torch / pyg   :', torch.__version__, '/', torch_geometric.__version__)
print()
print('similarity blocks (index order matters for every gate readout below):')
for b, (et, nt) in enumerate(DISEASE_SIM_BLOCKS):
    print('  %d  %-18s via %s' % (b, nt, et))
print()
print('agg_measures available:', ', '.join(VALID_AGG_MEASURES))

In [ ]:
def set_all_seeds(seed: int):
    '''Seed every RNG the training path touches.'''
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
if DEVICE == 'cpu':
    print('WARNING: CPU-only. Use PROFILE = "smoke".')

## 2 · Configuration

`ARMS` drives everything. Drop an entry to skip that arm; arm A is read from disk when
`REUSE_BASELINE_RESULTS` is on, so leaving it in costs nothing.

The Gumbel hyperparameters below are the ones the redesign added. `gumbel_anneal_steps` should
track `finetune.n_epoch`: fine-tuning is **full-batch — one optimizer step per epoch** — so
1000 epochs is 1000 gradient updates, and the temperature has to complete its anneal within them.

In [ ]:
PROFILE = 'full'             # 'smoke' to validate the plumbing, 'full' to get real numbers
SPLIT   = 'complex_disease'  # zero-shot protocol, same as the baseline notebook
SEED    = 1

# ── shared model settings (identical across arms) ───────────────────────────────
BASE_MODEL_CFG = dict(
    n_hid = 100, n_inp = 100, n_out = 100,
    proto = True,
    proto_num = 3,
    attention = False,
    sim_measure = 'all_nodes_profile',
    exp_lambda = 0.7,        # now persisted in config.pkl (it silently reset to 0.7 before)
)

# ── the arms ────────────────────────────────────────────────────────────────────
# 'gumbel_*' keys are ignored by the non-Gumbel arms but are still written to config.pkl.
ARMS = {
    'A_rarity': dict(
        label = 'A · baseline (rarity, 2 blocks)',
        cfg   = dict(BASE_MODEL_CFG, agg_measure='rarity'),
    ),
    'B_rarity_4block': dict(
        label = 'B · 4 blocks, fixed gate',
        cfg   = dict(BASE_MODEL_CFG, agg_measure='rarity_4block'),
    ),
    'C_gumbel_block': dict(
        label = 'C · 4 blocks, learned Gumbel gate',
        cfg   = dict(BASE_MODEL_CFG, agg_measure='gumbel_block',
                     gumbel_tau_start      = 1.0,
                     gumbel_tau_end        = 0.3,
                     gumbel_anneal_steps   = 1000,   # == finetune n_epoch for the full profile
                     gumbel_hidden         = 64,
                     gumbel_entropy_weight = 0.01),
    ),
}

_PROFILES = {
    'smoke': dict(
        pretrain = dict(n_epoch=0,  learning_rate=1e-3, batch_size=1024, train_print_per_n=50),
        finetune = dict(n_epoch=20, learning_rate=5e-4, train_print_per_n=5, valid_per_n=10),
        eval     = dict(n_diseases=5, simulate_random=False),
    ),
    'full': dict(
        pretrain = dict(n_epoch=1,    learning_rate=1e-3, batch_size=1024, train_print_per_n=100),
        finetune = dict(n_epoch=1000, learning_rate=5e-4, train_print_per_n=50, valid_per_n=20),
        eval     = dict(n_diseases=None, simulate_random=True),
    ),
}
PROF = _PROFILES[PROFILE]

# ── reuse switches ──────────────────────────────────────────────────────────────
REUSE_PRETRAINED        = True   # §4 — share one pretrained encoder across arms
REUSE_BASELINE_RESULTS  = True   # §7 — read arm A from the baseline notebook's results/
RUN_MULTISEED           = False  # §9 — repeat every arm over 5 seeds (expensive)
SEEDS                   = [1, 2, 3, 4, 5]

# ── paths (kept separate from the baseline notebook's outputs) ──────────────────
DATA_FOLDER   = REPO_ROOT / 'data'
GUMBEL_RESULTS = REPO_ROOT / 'results' / 'gumbel'
GUMBEL_MODELS  = REPO_ROOT / 'saved_models' / 'gumbel'
SHARED_PRETRAIN = REPO_ROOT / 'saved_models' / 'shared_pretrained'
for d in (GUMBEL_RESULTS, GUMBEL_MODELS, SHARED_PRETRAIN):
    d.mkdir(parents=True, exist_ok=True)

def run_name(arm, seed=SEED, split=SPLIT):
    return f'{arm}_{split}_seed{seed}_{PROFILE}'

def result_dir(arm, seed=SEED):
    d = GUMBEL_RESULTS / run_name(arm, seed); d.mkdir(parents=True, exist_ok=True); return d

def model_dir(arm, seed=SEED):
    d = GUMBEL_MODELS / run_name(arm, seed); d.mkdir(parents=True, exist_ok=True); return d

if PROFILE == 'full' and ARMS.get('C_gumbel_block'):
    n_ep = PROF['finetune']['n_epoch']
    n_an = ARMS['C_gumbel_block']['cfg']['gumbel_anneal_steps']
    if n_an > n_ep:
        print(f'WARNING: gumbel_anneal_steps={n_an} > finetune n_epoch={n_ep}. '
              f'tau will never reach tau_end.')

print('profile  :', PROFILE, '| split:', SPLIT, '| seed:', SEED)
print('arms     :', ', '.join(ARMS))
print('finetune :', PROF['finetune'])
print('results  ->', GUMBEL_RESULTS)
print('models   ->', GUMBEL_MODELS)

## 3 · Data and splits

Same cached splits the baseline notebook built — nothing is recomputed if `data/<split>_<seed>/` exists.

In [ ]:
set_all_seeds(SEED)
t0 = time.time()
txdata = TxData(data_folder_path=str(DATA_FOLDER))
txdata.prepare_split(split=SPLIT, seed=SEED, no_kg=False)
print('\nprepare_split took %.1f s' % (time.time() - t0))

DD_REL     = ['contraindication', 'indication', 'off-label use']
DD_REV_REL = ['rev_' + r for r in DD_REL]
DD_ETYPES  = [('drug', r, 'disease') for r in DD_REL] + [('disease', r, 'drug') for r in DD_REV_REL]

print()
for rel in DD_REL:
    print('%-20s train %7d | valid %6d | test %6d' % (
        rel,
        int((txdata.df_train.relation == rel).sum()),
        int((txdata.df_valid.relation == rel).sum()),
        int((txdata.df_test.relation  == rel).sum())))

## 4 · Shared pretrained encoder

**Why one encoder is valid for all three arms.** `pretrain()` calls the predictor with
`pretrain_mode=True`, which returns before the disease-similarity block entirely. No arm's gate
sees a gradient during pretraining, and no arm's gate parameters affect the encoder. The three
arms would therefore each pretrain to the *same* encoder — so sharing one is not an
approximation, it removes a source of variance and ~19 min per arm.

The search order below prefers a checkpoint your baseline run already wrote.

In [ ]:
def find_pretrained(seed=SEED, split=SPLIT):
    '''Locate a reusable pretrained encoder, preferring the baseline notebook's own checkpoint.'''
    candidates = [
        SHARED_PRETRAIN / f'{split}_seed{seed}',                                # written by us
        REPO_ROOT / 'saved_models' / f'TxGNN_{split}_seed{seed}_{PROFILE}' / 'pretrained',
        REPO_ROOT / 'saved_models' / f'TxGNN_{split}_seed{seed}_full' / 'pretrained',
    ]
    for c in candidates:
        if (c / 'model.pt').exists():
            return c, candidates
    return None, candidates


PRETRAIN_CKPT, _cands = find_pretrained()
print('searched:')
for c in _cands:
    print('  %-70s %s' % (c.relative_to(REPO_ROOT), 'FOUND' if (c / "model.pt").exists() else '-'))
print()
if PRETRAIN_CKPT is not None:
    sd_probe = torch.load(PRETRAIN_CKPT / 'model.pt', map_location='cpu')
    print('reusing pretrained encoder ->', PRETRAIN_CKPT.relative_to(REPO_ROOT))
    print('  %d tensors, %s parameters' % (
        len(sd_probe), f'{sum(v.numel() for v in sd_probe.values()):,}'))
    print('  keys:', ', '.join(sorted(sd_probe)[:6]), '...')
    del sd_probe
elif not REUSE_PRETRAINED or PROF['pretrain']['n_epoch'] == 0:
    print('No pretrained encoder; PROFILE=%r will fine-tune from random encoder weights.' % PROFILE)
else:
    print('No pretrained encoder found. The next cell will create one (~19 min, once).')

In [ ]:
# Creates the shared encoder only if none was found. Safe to re-run: it is a no-op afterwards.
if PRETRAIN_CKPT is None and REUSE_PRETRAINED and PROF['pretrain']['n_epoch'] > 0:
    set_all_seeds(SEED)
    _pre = TxGNN(data=txdata, weight_bias_track=False,
                 exp_name='shared_pretrain', device=DEVICE)
    # Architecture-independent: any arm's config yields the same encoder (see the note above).
    _pre.model_initialize(**ARMS['A_rarity']['cfg'])

    t0 = time.time()
    set_all_seeds(SEED)
    _pre.pretrain(**PROF['pretrain'])
    print('\npretraining took %.1f min' % ((time.time() - t0) / 60))

    PRETRAIN_CKPT = SHARED_PRETRAIN / f'{SPLIT}_seed{SEED}'
    _pre.save_model(str(PRETRAIN_CKPT))
    print('shared encoder ->', PRETRAIN_CKPT.relative_to(REPO_ROOT))

    del _pre
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    print('nothing to do')

## 5 · Fine-tuning each arm

`load_pretrained()` cannot be used here: it calls `model_initialize(**saved_config)`, which would
rebuild the model with the *checkpoint's* `agg_measure` (`rarity`) instead of the arm's. Instead we
build the arm's architecture first and load the encoder tensors into it with `strict=False`.

The load is checked rather than trusted. A pretrained checkpoint contains encoder weights and
`w_rels` only — the gate parameters do not exist in it — so exactly two things are acceptable:
**no unexpected keys at all**, and **every missing key belongs to a gate**. Anything else means the
checkpoint and the architecture disagree, and the cell raises instead of silently training a
half-initialised model.

In [ ]:
GATE_KEY_MARKERS = ('gate_mlps', 'gumbel_step', 'W_gate')

def load_encoder_into(tx, ckpt_dir, verbose=True):
    '''Load encoder weights from a pretrained checkpoint into an already-built model.

    Returns the list of missing keys (the gate parameters, left at their fresh init).
    Raises if the checkpoint contains keys the model does not have, or if a non-gate key
    is missing -- either means the encoder did not actually land.
    '''
    sd = torch.load(Path(ckpt_dir) / 'model.pt', map_location='cpu')
    if next(iter(sd)).startswith('module.'):
        sd = OrderedDict((k[7:], v) for k, v in sd.items())

    # A concrete encoder tensor, snapshotted so we can prove the load actually changed it.
    probe_key = sorted(tx.model.layer1.weight.keys())[0]
    before = tx.model.layer1.weight[probe_key].weight.detach().clone()

    incompat = tx.model.load_state_dict(sd, strict=False)
    missing, unexpected = list(incompat.missing_keys), list(incompat.unexpected_keys)

    if unexpected:
        raise RuntimeError(
            'Checkpoint has %d key(s) this architecture does not: %s\n'
            'The checkpoint and the model disagree -- do not train on this.'
            % (len(unexpected), unexpected[:8]))

    bad = [k for k in missing if not any(m in k for m in GATE_KEY_MARKERS)]
    if bad:
        raise RuntimeError(
            'These non-gate weights were NOT loaded: %s\n'
            'The encoder is only partially initialised.' % bad[:8])

    # Positive confirmation: the probed encoder tensor must have moved off its fresh init.
    after = tx.model.layer1.weight[probe_key].weight.detach()
    if torch.equal(before, after):
        raise RuntimeError(
            'Encoder weights are unchanged after load (probe: layer1.weight[%r]). '
            'The checkpoint did not take effect.' % probe_key)

    if verbose:
        print('  loaded   %d tensors from %s' % (len(sd), Path(ckpt_dir).name))
        print('  encoder  changed (probe layer1.weight[%r], max |delta| %.4g)'
              % (probe_key, (after - before).abs().max().item()))
        print('  missing  %d (all gate params, expected): %s'
              % (len(missing), ', '.join(missing) if missing else 'none'))
    tx.model = tx.model.to(tx.device)
    tx.best_model = tx.model
    return missing


def build_arm(arm, seed=SEED, use_pretrained=True):
    '''Build one arm's model, optionally warm-started from the shared encoder.'''
    set_all_seeds(seed)
    tx = TxGNN(data=txdata, weight_bias_track=False,
               exp_name=run_name(arm, seed), device=DEVICE)
    tx.model_initialize(**ARMS[arm]['cfg'])

    if use_pretrained and PRETRAIN_CKPT is not None:
        load_encoder_into(tx, PRETRAIN_CKPT)
    else:
        print('  (no pretrained encoder -- random init)')
    return tx

print('helpers ready')

In [ ]:
def finetune_arm(arm, seed=SEED, force=False):
    '''Fine-tune one arm and persist it. Cached on disk: re-running is free.'''
    mdir, rdir = model_dir(arm, seed), result_dir(arm, seed)
    if (mdir / 'model.pt').exists() and not force:
        print(f'[{arm}] cached checkpoint -> {mdir.relative_to(REPO_ROOT)}')
        return None

    print('#' * 78)
    print('# %s  |  %s  |  seed %d' % (arm, ARMS[arm]['label'], seed))
    print('#' * 78)

    tx = build_arm(arm, seed)
    n_params = sum(p.numel() for p in tx.model.parameters())
    print('  parameters: %s' % f'{n_params:,}')

    pred = tx.model.pred
    if getattr(pred, 'block_mode', False):
        print('  sim_blocks: %s on %s (%.1f MB)' % (
            tuple(pred.sim_blocks.shape), pred.sim_blocks.device,
            pred.sim_blocks.numel() * pred.sim_blocks.element_size() / 1e6))
    else:
        for et, sim in list(pred.sim_all_etypes.items())[:1]:
            print('  sim_all_etypes: %d etypes, each %s' % (
                len(pred.sim_all_etypes), tuple(sim.shape)))

    set_all_seeds(seed)
    t0 = time.time()
    tx.finetune(**PROF['finetune'])
    secs = time.time() - t0
    print('\n  fine-tuning took %.1f min' % (secs / 60))

    if ARMS[arm]['cfg'].get('agg_measure') == 'gumbel_block':
        print('  gumbel_step reached %d (tau -> %.3f)' % (
            int(tx.model.pred.gumbel_step.item()), tx.model.pred._current_tau()))

    tx.save_model(str(mdir))
    with open(rdir / 'run_config.json', 'w') as f:
        json.dump({'arm': arm, 'label': ARMS[arm]['label'], 'seed': seed, 'split': SPLIT,
                   'profile': PROFILE, 'model': ARMS[arm]['cfg'], 'finetune': PROF['finetune'],
                   'pretrained_from': str(PRETRAIN_CKPT) if PRETRAIN_CKPT else None,
                   'n_params': int(n_params), 'finetune_seconds': secs,
                   'torch': torch.__version__, 'pyg': torch_geometric.__version__},
                  f, indent=2, default=str)
    return tx


TRAINED = {}
for arm in ARMS:
    if arm == 'A_rarity' and REUSE_BASELINE_RESULTS:
        print(f'[{arm}] skipped -- reading baseline from results/ (see section 7)\n')
        continue
    TRAINED[arm] = finetune_arm(arm)

print('\ndone:', list(TRAINED))

## 6 · Evaluation

Identical protocol to the baseline notebook: edge-level AUROC/AUPRC per relation, then the
paper's disease-centric ranking metrics. `summarise()` is copied verbatim so the two notebooks'
numbers mean the same thing.

In [ ]:
HEADLINE = ['AUPRC', 'AUROC',
            'Recall@1%', 'Recall@5%', 'Recall@10%', 'Recall@10', 'Recall@50', 'Recall@100',
            'MRR@10', 'MRR@50', 'MRR@100',
            'AP@10', 'AP@50', 'AP@100',
            'Enrichment@1%', 'Enrichment@5%', 'Enrichment@10%',
            'F1', 'Accuracy', 'Sensitivity', 'Specificity', '# of Pos']

def summarise(df, metrics=HEADLINE, drop_sentinels=True):
    '''Mean/std over diseases. Verbatim from the baseline notebook (D8).'''
    rows = {}
    for m in metrics:
        if m not in df.columns:
            continue
        v = pd.to_numeric(df[m], errors='coerce').replace([np.inf, -np.inf], np.nan)
        if drop_sentinels and m != '# of Pos':
            v = v[v != -1]
        v = v.dropna()
        rows[m] = {'mean': v.mean(), 'std': v.std(ddof=0), 'n_diseases': int(v.size)}
    return pd.DataFrame(rows).T


def load_arm(arm, seed=SEED):
    '''Rebuild an arm from its saved checkpoint (so evaluation can run without retraining).'''
    if arm in TRAINED and TRAINED[arm] is not None:
        return TRAINED[arm]
    mdir = model_dir(arm, seed)
    tx = TxGNN(data=txdata, weight_bias_track=False, exp_name=run_name(arm, seed), device=DEVICE)
    tx.load_pretrained(str(mdir))       # config.pkl carries this arm's agg_measure
    return tx


def evaluate_arm(arm, seed=SEED, force=False):
    '''Edge-level + disease-centric evaluation for one arm. Caches to results/gumbel/<run>/.'''
    rdir = result_dir(arm, seed)
    edge_csv, dc_csv = rdir / 'edge_level_test_metrics.csv', rdir / 'disease_centric_summary.csv'
    if edge_csv.exists() and dc_csv.exists() and not force:
        print(f'[{arm}] cached results')
        return pd.read_csv(edge_csv), pd.read_csv(dc_csv, index_col=0)

    from txgnn.utils import evaluate_fb
    tx = load_arm(arm, seed)
    G_dev = tx.G.to(DEVICE)

    # ── edge level ──
    (auroc_rel, auprc_rel, micro_auroc, micro_auprc, macro_auroc, macro_auprc), test_loss, _, _ = \
        evaluate_fb(tx.best_model, tx.g_test_pos, tx.g_test_neg,
                    G_dev, tx.dd_etypes, DEVICE, return_embed=True, mode='test')

    edge = pd.DataFrame(
        [(f'{s} --{r}--> {d}', auroc_rel.get((s, r, d), np.nan), auprc_rel.get((s, r, d), np.nan))
         for s, r, d in DD_ETYPES], columns=['relation', 'AUROC', 'AUPRC'])
    edge.loc[len(edge)] = ['MICRO (all relations)', micro_auroc, micro_auprc]
    edge.loc[len(edge)] = ['MACRO (all relations)', macro_auroc, macro_auprc]
    edge.to_csv(edge_csv, index=False)
    print(f'[{arm}] test BCE {test_loss:.4f} | macro AUPRC {macro_auprc:.4f}')

    # ── disease centric ──
    ev = TxEval(model=tx)
    t0 = time.time()
    if PROF['eval']['n_diseases'] is None:
        dc = ev.eval_disease_centric(disease_idxs='test_set', relation=None, save_result=True,
                                     save_name=str(rdir / 'disease_centric_eval.pkl'),
                                     show_plot=False, verbose=False, return_raw=False,
                                     simulate_random=PROF['eval']['simulate_random'])
    else:
        dc = {}
        for rel in DD_REL:
            idxs = ev.retrieve_disease_idxs_test_set(rel)[:PROF['eval']['n_diseases']]
            dc['rev_' + rel] = ev.eval_disease_centric(
                disease_idxs=list(idxs), relation=rel, save_result=False,
                show_plot=False, verbose=False, return_raw=False,
                simulate_random=PROF['eval']['simulate_random'])
        with open(rdir / 'disease_centric_eval.pkl', 'wb') as f:
            pickle.dump(dc, f)
    print(f'[{arm}] disease-centric eval took {(time.time()-t0)/60:.1f} min')

    summary = pd.DataFrame({r.replace('rev_', ''): summarise(df)['mean'] for r, df in dc.items()})
    summary.to_csv(dc_csv)
    return edge, summary

print('helpers ready')

In [ ]:
EDGE, DC = {}, {}

for arm in ARMS:
    if arm == 'A_rarity' and REUSE_BASELINE_RESULTS:
        base = REPO_ROOT / 'results' / f'TxGNN_{SPLIT}_seed{SEED}_{PROFILE}'
        if (base / 'edge_level_test_metrics.csv').exists():
            EDGE[arm] = pd.read_csv(base / 'edge_level_test_metrics.csv')
            DC[arm]   = pd.read_csv(base / 'disease_centric_summary.csv', index_col=0)
            print(f'[{arm}] baseline read from {base.relative_to(REPO_ROOT)}')
            continue
        print(f'[{arm}] no baseline results at {base.relative_to(REPO_ROOT)} -- training it')
        TRAINED[arm] = finetune_arm(arm)
    EDGE[arm], DC[arm] = evaluate_arm(arm)

print('\nevaluated:', list(EDGE))

## 7 · Arm-by-arm comparison

B isolates the effect of the two extra signature blocks; C adds the learned gate on top of B.
The honest read of "did the gate help" is **C − B**, not C − A.

In [ ]:
def edge_auprc(edge_df, rel):
    m = edge_df.relation.str.contains(f'--{rel}-->', regex=False)
    return float(edge_df.loc[m, 'AUPRC'].iloc[0]) if m.any() else np.nan

rows = []
for arm in EDGE:
    r = {'arm': ARMS[arm]['label']}
    for rel in DD_REL:
        r[rel] = edge_auprc(EDGE[arm], rel)
    r['MACRO'] = float(EDGE[arm].loc[EDGE[arm].relation.str.startswith('MACRO'), 'AUPRC'].iloc[0])
    rows.append(r)

edge_cmp = pd.DataFrame(rows).set_index('arm')
print('EDGE-LEVEL AUPRC (test set)')
display(edge_cmp.style.format('{:.4f}', na_rep='n/a'))

if len(edge_cmp) > 1:
    delta = edge_cmp - edge_cmp.iloc[0]
    print('\ndelta vs %s' % edge_cmp.index[0])
    display(delta.iloc[1:].style.format('{:+.4f}', na_rep='n/a')
                 .background_gradient(cmap='RdYlGn', vmin=-0.05, vmax=0.05))

In [ ]:
SHOW = ['AUPRC', 'AUROC', 'Recall@1%', 'Recall@5%', 'Recall@10%', 'MRR@50', 'AP@50']

for rel in DD_REL:
    tbl = pd.DataFrame({ARMS[a]['label']: DC[a][rel] for a in DC if rel in DC[a].columns})
    tbl = tbl.loc[[m for m in SHOW if m in tbl.index]]
    print('\n' + '=' * 78)
    print('DISEASE-CENTRIC  ·  %s' % rel.upper())
    print('=' * 78)
    display(tbl.style.format('{:.4f}'))
    if tbl.shape[1] > 1:
        d = tbl.sub(tbl.iloc[:, 0], axis=0).iloc[:, 1:]
        display(d.style.format('{:+.4f}')
                 .background_gradient(cmap='RdYlGn', vmin=-0.05, vmax=0.05))

In [ ]:
# One figure: disease-centric AUPRC per relation, arms side by side.
metrics = [m for m in ['AUPRC', 'Recall@5%', 'MRR@50'] if all(m in DC[a].index for a in DC)]
fig, axes = plt.subplots(1, len(metrics), figsize=(5.2 * len(metrics), 4), squeeze=False)
x = np.arange(len(DD_REL)); w = 0.8 / max(len(DC), 1)

for ax, metric in zip(axes[0], metrics):
    for i, arm in enumerate(DC):
        vals = [DC[arm][rel].get(metric, np.nan) if rel in DC[arm].columns else np.nan
                for rel in DD_REL]
        ax.bar(x + i * w - 0.4 + w / 2, vals, w, label=ARMS[arm]['label'].split(' · ')[0])
    ax.set_xticks(x); ax.set_xticklabels([r.replace(' ', '\n') for r in DD_REL], fontsize=9)
    ax.set_title(metric); ax.grid(axis='y', alpha=0.3)
axes[0][0].set_ylabel('disease-centric score')
axes[0][-1].legend(fontsize=8, loc='best')
plt.tight_layout(); plt.show()

## 8 · What did the gate actually learn?

The point of the redesign is per-disease, per-block *selection*, so the headline metrics are only
half the story — this section reads the decisions out directly.

`record_gates` makes the predictor log every `(etype, disease_ids, gate[N_q, 4])` it produces. We
run one forward over the **positive test graph**, which for a zero-shot split also exercises the
unseen-disease fallback path on real data.

In [ ]:
GATE_ARM = 'C_gumbel_block'

if GATE_ARM not in ARMS:
    print('Gumbel arm not enabled -- nothing to inspect.')
    gate_df = None
else:
    tx = load_arm(GATE_ARM)
    pred = tx.best_model.pred
    G_dev = tx.G.to(DEVICE)

    tx.best_model.eval()
    pred.gate_log, pred.record_gates = [], True
    with torch.no_grad():
        h = tx.best_model(G_dev, G_dev, return_h=True)
        pred(tx.g_test_pos, G_dev, h, False, mode='test_pos')
    pred.record_gates = False

    recs = []
    for etype, ids, gate in pred.gate_log:
        for j, d in enumerate(ids):
            row = {'etype': '%s->%s' % (etype[1], etype[2]), 'relation': etype[1],
                   'disease_idx': d}
            for b in range(N_SIM_BLOCKS):
                row[BLOCK_NODES[b]] = float(gate[j, b])
            recs.append(row)
    gate_df = pd.DataFrame(recs)
    print('logged %d gate decisions across %d etypes, %d distinct diseases'
          % (len(gate_df), gate_df.relation.nunique(), gate_df.disease_idx.nunique()))
    display(gate_df.head())

In [ ]:
if gate_df is not None and len(gate_df):
    print('BLOCK SELECTION RATE  (fraction of diseases for which the block is used)\n')
    usage = gate_df.groupby('relation')[BLOCK_NODES].mean()
    usage.loc['ALL'] = gate_df[BLOCK_NODES].mean()
    display(usage.style.format('{:.3f}').background_gradient(cmap='Blues', vmin=0, vmax=1))

    n_open = gate_df[BLOCK_NODES].sum(axis=1)
    print('\nBLOCKS OPEN PER DISEASE')
    dist = n_open.value_counts().sort_index().rename('diseases').to_frame()
    dist['fraction'] = dist['diseases'] / len(n_open)
    display(dist.style.format({'diseases': '{:.0f}', 'fraction': '{:.3f}'}))
    print('mean blocks open: %.2f of %d' % (n_open.mean(), N_SIM_BLOCKS))
    if (n_open == 0).any():
        print('%d decision(s) opened NO block -> fell back to the disease\'s own embedding'
              % int((n_open == 0).sum()))
    if n_open.std() < 1e-6:
        print('\nWARNING: the gate is constant across diseases -- it collapsed. '
              'Consider a longer anneal, a lower tau_end, or a larger gumbel_entropy_weight.')

In [ ]:
if gate_df is not None and len(gate_df):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    usage.drop(index='ALL', errors='ignore').plot.bar(ax=axes[0], rot=0, width=0.8)
    axes[0].set_ylim(0, 1); axes[0].set_ylabel('selection rate')
    axes[0].set_title('Block selection rate by relation'); axes[0].grid(axis='y', alpha=0.3)
    axes[0].legend(fontsize=8)

    combo = (gate_df[BLOCK_NODES].astype(int).astype(str).agg(''.join, axis=1)
             .value_counts().head(10) / len(gate_df))
    combo.plot.barh(ax=axes[1])
    axes[1].set_xlabel('fraction of diseases')
    axes[1].set_title('Most common block combinations\n(bit order: %s)'
                      % ', '.join(BLOCK_NODES))
    axes[1].grid(axis='x', alpha=0.3)
    plt.tight_layout(); plt.show()

In [ ]:
# Does the learned gate rediscover the degree signal the fixed rarity gate hard-coded?
if gate_df is not None and len(gate_df):
    from torch_geometric.utils import degree as pyg_degree

    tx = load_arm(GATE_ARM)
    G_cpu = tx.G.to('cpu')
    deg = {}
    for etype in tx.dd_etypes:
        if etype not in G_cpu.edge_types:
            continue
        if etype[0] == 'disease':
            d = pyg_degree(G_cpu[etype].edge_index[0], num_nodes=G_cpu['disease'].num_nodes)
        else:
            d = pyg_degree(G_cpu[etype].edge_index[1], num_nodes=G_cpu['disease'].num_nodes)
        for i, v in enumerate(d.tolist()):
            deg[(etype[1], i)] = v

    g = gate_df.copy()
    g['degree'] = [deg.get((r, i), np.nan) for r, i in zip(g.relation, g.disease_idx)]
    g['n_open'] = g[BLOCK_NODES].sum(axis=1)
    g = g.dropna(subset=['degree'])

    if len(g) and g.degree.nunique() > 1:
        print('Spearman correlation of block selection with the disease\'s drug-disease degree')
        print('(the fixed rarity gate is a decreasing function of exactly this quantity)\n')
        corr = pd.Series({b: g[[b, 'degree']].corr(method='spearman').iloc[0, 1]
                          for b in BLOCK_NODES + ['n_open']}, name='spearman rho')
        display(corr.to_frame().style.format('{:+.3f}')
                    .background_gradient(cmap='coolwarm', vmin=-0.5, vmax=0.5))
        print('Near-zero => the gate learned something orthogonal to rarity.')
        print('Strongly negative => it re-derived "trust prototypes more for rare diseases".')
    else:
        print('Not enough degree variation among test diseases to correlate '
              '(expected on a zero-shot split, where held-out diseases have degree 0).')

## 9 · Multi-seed

Repeats every arm over 5 splits and reports mean ± 95% CI, matching §9 of the baseline notebook.
Cost is roughly (number of arms × seeds) fine-tuning runs, so it is off by default.

In [ ]:
def run_arm_seed(arm, seed):
    '''Full pipeline for one (arm, seed). Cached on disk.'''
    rdir = result_dir(arm, seed)
    out_csv = rdir / 'disease_centric_summary.csv'
    if out_csv.exists():
        return pd.read_csv(out_csv, index_col=0)

    global txdata, PRETRAIN_CKPT
    set_all_seeds(seed)
    txdata = TxData(data_folder_path=str(DATA_FOLDER))
    txdata.prepare_split(split=SPLIT, seed=seed, no_kg=False)

    PRETRAIN_CKPT, _ = find_pretrained(seed=seed)
    if PRETRAIN_CKPT is None and PROF['pretrain']['n_epoch'] > 0:
        set_all_seeds(seed)
        p = TxGNN(data=txdata, weight_bias_track=False, device=DEVICE)
        p.model_initialize(**ARMS['A_rarity']['cfg'])
        set_all_seeds(seed); p.pretrain(**PROF['pretrain'])
        PRETRAIN_CKPT = SHARED_PRETRAIN / f'{SPLIT}_seed{seed}'
        p.save_model(str(PRETRAIN_CKPT))
        del p
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    finetune_arm(arm, seed)
    _, summary = evaluate_arm(arm, seed)
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return summary


if RUN_MULTISEED:
    multi = {}
    for arm in ARMS:
        per_seed = {s: run_arm_seed(arm, s) for s in SEEDS}
        stacked = pd.concat(per_seed, names=['seed'])
        agg = stacked.groupby(level=1).agg(['mean', 'std', 'count'])
        ci = {}
        for rel in per_seed[SEEDS[0]].columns:
            m_, s_, n_ = agg[(rel, 'mean')], agg[(rel, 'std')], agg[(rel, 'count')]
            ci[rel] = m_.round(4).astype(str) + ' ± ' + (1.96 * s_ / np.sqrt(n_)).round(4).astype(str)
        multi[arm] = pd.DataFrame(ci)
        multi[arm].to_csv(GUMBEL_RESULTS / f'multiseed_{arm}_{SPLIT}_{PROFILE}.csv')

        print('\n' + '=' * 78)
        print('%s — mean ± 95%% CI over %d splits' % (ARMS[arm]['label'], len(SEEDS)))
        print('=' * 78)
        display(multi[arm].loc[[i for i in SHOW if i in multi[arm].index]])
else:
    print('RUN_MULTISEED is False — single-seed results only (see section 2 to enable).')

## 10 · Reading these results

**Attribute gains to the right change.** A gain of C over A confounds two things: four signature
blocks instead of two, and a learned gate instead of a fixed one. Arm B exists precisely to split
them — report **B − A** as "more signal" and **C − B** as "smarter gating".

**Per-block cosine is not the old cosine.** The baseline takes one cosine over the *concatenated*
signature, which implicitly weights each block by its L2 mass — a disease with many protein
neighbours has its protein block dominate. Arms B and C take four separately-normalised cosines.
So even B with all gates forced open would not reproduce A exactly; the similarity metric itself
changed, not only the gating.

**The gate has ~1000 gradient updates, total.** Fine-tuning is full-batch: one optimizer step per
epoch, no node sampling, no dropout anywhere in the encoder, and the gate receives *no* gradient
during pretraining. Its only stochasticity is the Gumbel noise itself. If §8 shows the gate is
constant across diseases, it collapsed — lengthen the anneal, raise `gumbel_entropy_weight`, or
lower `tau_end`, and re-check §8 before reading anything into the headline metrics.

**Zero-shot diseases take a different code path.** On `complex_disease` the held-out diseases have
no drug–disease edge in the training graph, so they are absent from the precomputed `[4, D, D]`
similarity tensor and their signatures are built on demand at eval time. §8 exercises that path;
if it raises, the headline numbers for this split are not trustworthy.

**Known gaps.**

- `retrieve_sim_diseases()` (§8.1 of the baseline notebook) raises `NotImplementedError` under a
  per-block `agg_measure`. Per-block matrices live on `model.pred.sim_blocks` with
  `model.pred.block_diseaseid2id` as the index map.
- The GraphMask Explainer runs through the new gate with the temperature pinned to `tau_end` and
  the step counter frozen, so explanations are never taken mid-anneal. It has not been run
  end-to-end against a Gumbel checkpoint.
- Block index 0 is **disease-disease**, not gene/protein — the order follows the pre-existing
  `disease_etypes_all` list. `BLOCK_NODES` printed in §1 is the authority for every readout here.

**Provenance.** Every arm writes `results/gumbel/<run>/run_config.json` recording its full model
config, the pretrained checkpoint it started from, and wall-clock time.